# 03 · Overlap between the Stage-1 and Stage-2 partitions

Notebook 01 (Stage 1) and notebook 02 (Stage 2) each draw their own 70/15/15 stratified split with seed 42, but over
different row sets (16,451 samples with a CFP or UWF image vs all 17,224 samples). The partitions therefore differ.
This notebook re-creates both splits with the exact code of notebooks 01 and 02 and counts how many Stage-2
validation and test samples were used to train the Stage-1 encoders. It needs only the CSV files (no GPU, no images).

It also verifies that the re-created Stage-2 test order is the one used in notebook 02: a gate weight is exactly 0
when its modality is missing, so the zero pattern of `results/approved_fast_attention_gnn_gate_weights.csv` must
match the modality presence of the re-created test rows.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/MMRDR_Project")
except ImportError:
    PROJECT_ROOT = Path(".").resolve()
DATA_ROOT = PROJECT_ROOT / "DatasetRT"
RESULTS = PROJECT_ROOT / "results"      # this repository's results/ folder
SEED = 42

manifest = pd.read_csv(DATA_ROOT / "manifest.csv")
metadata = pd.read_csv(DATA_ROOT / "metadata.csv")
base = manifest.merge(metadata, on=["sample_id", "patient_id"], how="left")
base = base.dropna(subset=["dr_grade"]).copy()
base["dr_grade"] = base["dr_grade"].astype(int)
base["referable_dr"] = (base["dr_grade"] >= 2).astype(int)

def two_step_split(df):
    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["referable_dr"])
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["referable_dr"])
    return train_df, val_df, test_df

# Stage 1 (notebook 01): samples with a CFP or UWF image
s1 = two_step_split(base[base["fundus_path"].notna() | base["uwf_path"].notna()].copy())
# Stage 2 (notebook 02): samples with any image
s2 = two_step_split(base[base["fundus_path"].notna() | base["oct_path"].notna() | base["uwf_path"].notna()].copy())

for name, parts in [("Stage 1", s1), ("Stage 2", s2)]:
    print(name, [(len(p), int(p["referable_dr"].sum())) for p in parts], "(n, referable) for train/val/test")

In [ ]:
key = lambda d: set(zip(d["sample_id"], d["eye"]))
stage1_train = key(s1[0])
rows = []
for name, part in [("validation", s2[1]), ("test", s2[2])]:
    eligible = part[part["fundus_path"].notna() | part["uwf_path"].notna()]
    overlap = len(key(part) & stage1_train)
    rows.append({"Stage-2 partition": name, "samples": len(part), "with CFP or UWF": len(eligible),
                 "in Stage-1 training partition": overlap, "% of partition": round(100 * overlap / len(part), 1)})
overlap_df = pd.DataFrame(rows)
display(overlap_df)

In [ ]:
gates_path = RESULTS / "approved_fast_attention_gnn_gate_weights.csv"
if gates_path.exists():
    gates = pd.read_csv(gates_path)
    test = s2[2].reset_index(drop=True)
    match = (((gates["fundus_gate"] > 0) == test["fundus_path"].notna()) &
             ((gates["oct_gate"] > 0) == test["oct_path"].notna()) &
             ((gates["uwf_gate"] > 0) == test["uwf_path"].notna()))
    print(f"Gate-pattern check: {int(match.sum())} of {len(match)} test rows match")